# Adaptive RAG with Ollama and Ollama Cloud

This notebook implements Adaptive RAG with a graph workflow:
- Route query to direct answer, vector retrieval, or web retrieval
- Grade retrieved docs
- Fallback to web retrieval when local docs are weak
- Generate final grounded answer

It uses `adaptiveRAG.pdf` by default and supports infinite user queries.

In [ ]:
# Section 1: Install dependencies (run once)
# %pip install -U langchain langchain-community langchain-ollama langgraph faiss-cpu pypdf python-dotenv tavily-python

In [ ]:
# Section 2: Imports
import os
from typing import TypedDict, List, Literal
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import StateGraph, START, END

In [ ]:
# Section 3: Environment and model configuration
load_dotenv()

LOCAL_OLLAMA_BASE_URL = os.getenv("LOCAL_OLLAMA_BASE_URL", "http://localhost:11434")
LOCAL_LLM_MODEL = os.getenv("LOCAL_LLM_MODEL", "openvoid/Void-Gemini:latest")
LOCAL_EMBED_MODEL = os.getenv("LOCAL_EMBED_MODEL", "nomic-embed-text")

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "")

In [ ]:
# Section 4: Initialize LLMs and tools
local_llm = ChatOllama(model=LOCAL_LLM_MODEL, base_url=LOCAL_OLLAMA_BASE_URL, temperature=0)

answer_llm = local_llm
embeddings = OllamaEmbeddings(model=LOCAL_EMBED_MODEL, base_url=LOCAL_OLLAMA_BASE_URL)

web_search = TavilySearchResults(max_results=5) if TAVILY_API_KEY else None

print("Answer LLM:", f"local_ollama ({LOCAL_LLM_MODEL})")
print("Web tool enabled:", bool(web_search))

In [ ]:
# Section 5: Build PDF retriever
PDF_PATH = input("Enter PDF path (default e:\\LangGraph\\RAGs\\AdaptiveRAG\\adaptiveRAG.pdf): " ).strip() or r"e:\LangGraph\RAGs\AdaptiveRAG\adaptiveRAG.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print(f"Loaded pages: {len(docs)}")
print(f"Total chunks: {len(chunks)}")

In [ ]:
# Section 6: Adaptive RAG state and nodes
class AdaptiveRAGState(TypedDict, total=False):
    query: str
    route: Literal["direct_answer", "vector_retrieve", "web_retrieve"]
    docs: List[Document]
    web_context: str
    answer: str


def route_query(state: AdaptiveRAGState) -> AdaptiveRAGState:
    query = state["query"]
    prompt = ChatPromptTemplate.from_template(
        """
Choose one route for the query.
Return exactly one token:
- direct_answer
- vector_retrieve
- web_retrieve

Routing rules:
- direct_answer: generic greeting/smalltalk/basic common info
- vector_retrieve: likely answerable from local PDF
- web_retrieve: needs fresh or external knowledge

Query: {query}
""".strip()
    )
    msg = prompt.format_messages(query=query)
    raw = answer_llm.invoke(msg).content.strip().lower()

    if "web_retrieve" in raw:
        route = "web_retrieve"
    elif "direct_answer" in raw:
        route = "direct_answer"
    else:
        route = "vector_retrieve"

    return {"route": route}


def direct_answer(state: AdaptiveRAGState) -> AdaptiveRAGState:
    query = state["query"]
    prompt = ChatPromptTemplate.from_template("Answer clearly in 5 to 8 lines.\nQuery: {query}")
    msg = prompt.format_messages(query=query)
    ans = answer_llm.invoke(msg).content
    return {"answer": ans}


def vector_retrieve(state: AdaptiveRAGState) -> AdaptiveRAGState:
    query = state["query"]
    docs = retriever.invoke(query)
    return {"docs": docs}


def grade_docs(state: AdaptiveRAGState) -> AdaptiveRAGState:
    query = state["query"]
    docs = state.get("docs", [])

    if not docs:
        return {"route": "web_retrieve"}

    grade_prompt = ChatPromptTemplate.from_template(
        """
Judge if these snippets are enough to answer the query well.
Return exactly one token:
- vector_retrieve
- web_retrieve

Query:
{query}

Snippets:
{snippets}
""".strip()
    )

    snippets = "\n\n".join(d.page_content[:700] for d in docs[:4])
    msg = grade_prompt.format_messages(query=query, snippets=snippets)
    raw = answer_llm.invoke(msg).content.strip().lower()
    decision = "web_retrieve" if "web_retrieve" in raw else "vector_retrieve"
    return {"route": decision}


def web_retrieve(state: AdaptiveRAGState) -> AdaptiveRAGState:
    if not web_search:
        return {"web_context": "Web search disabled: TAVILY_API_KEY not set."}

    query = state["query"]
    results = web_search.invoke(query)

    if isinstance(results, list):
        lines = []
        for item in results:
            if isinstance(item, dict):
                title = item.get("title", "")
                content = item.get("content", "")
                url = item.get("url", "")
                lines.append(f"Title: {title}\nURL: {url}\nContent: {content}")
        context = "\n\n".join(lines)
    else:
        context = str(results)

    return {"web_context": context}


def generate_answer(state: AdaptiveRAGState) -> AdaptiveRAGState:
    query = state["query"]
    route = state.get("route", "vector_retrieve")
    doc_context = "\n\n".join(d.page_content for d in state.get("docs", []))
    web_context = state.get("web_context", "")

    prompt = ChatPromptTemplate.from_template(
        """
You are an Adaptive RAG assistant.
Use the available context based on route: {route}

If route is vector_retrieve, rely on Document Context.
If route is web_retrieve, rely on Web Context.
If route is direct_answer, answer directly.

If context is insufficient, explicitly mention limitations.

Query:
{query}

Document Context:
{doc_context}

Web Context:
{web_context}
""".strip()
    )

    msg = prompt.format_messages(
        route=route,
        query=query,
        doc_context=doc_context,
        web_context=web_context,
    )
    ans = answer_llm.invoke(msg).content
    return {"answer": ans}


def route_after_query(state: AdaptiveRAGState) -> str:
    return state.get("route", "vector_retrieve")


def route_after_grade(state: AdaptiveRAGState) -> str:
    return state.get("route", "vector_retrieve")

In [ ]:
# Section 7: Build Adaptive RAG graph
builder = StateGraph(AdaptiveRAGState)

builder.add_node("route_query", route_query)
builder.add_node("direct_answer", direct_answer)
builder.add_node("vector_retrieve", vector_retrieve)
builder.add_node("grade_docs", grade_docs)
builder.add_node("web_retrieve", web_retrieve)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "route_query")
builder.add_conditional_edges(
    "route_query",
    route_after_query,
    {
        "direct_answer": "direct_answer",
        "vector_retrieve": "vector_retrieve",
        "web_retrieve": "web_retrieve",
    },
)

builder.add_edge("direct_answer", END)
builder.add_edge("vector_retrieve", "grade_docs")
builder.add_conditional_edges(
    "grade_docs",
    route_after_grade,
    {
        "vector_retrieve": "generate_answer",
        "web_retrieve": "web_retrieve",
    },
)
builder.add_edge("web_retrieve", "generate_answer")
builder.add_edge("generate_answer", END)

app = builder.compile()
app

In [25]:
# Section 8: Infinite query loop
while True:
    user_query = input("\nAsk your question (or type 'exit'): " ).strip()
    if user_query.lower() in {"exit", "quit", "q"}:
        print("Stopped.")
        break
    if not user_query:
        print("Please enter a valid query.")
        continue

    state_out = app.invoke({"query": user_query})

    print("\n=== Route Used ===")
    print(state_out.get("route", "unknown"))

    print("\n=== Final Answer ===")
    print(state_out.get("answer", "No answer generated."))


=== Route Used ===
web_retrieve

=== Final Answer ===
Based on the provided web context, the current Prime Minister of Nepal is **Balendra Shah** (popularly known as **Balen**).

Key details regarding his premiership include:
*   **Appointment Date:** He was sworn in on **March 27, 2026**, by President Ram Chandra Paudel.
*   **Political Background:** A former rapper and structural engineer, he led the **Rastriya Swatantra Party (RSP)** to a landslide victory in the general elections held on March 5, 2024.
*   **Significance:** At age 35, he is the youngest Prime Minister in Nepal's recent political history. His rise followed a period of political transition triggered by youth-led protests that overthrew the previous government of K.P. Sharma Oli.
*   **Official Residence:** His official residence is in Baluwatar, Kathmandu, and his office is located at Singha Darbar.

**Note on Context:** While some older documents in the context mention Pushpa Kamal Dahal or K.P. Sharma Oli as the P